In [24]:
# Import the libraries needed for data cleaning and transformation

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

In [2]:
# Verify Pandas and PyArrow load correctly after restarting the kernel

import pandas as pd
import pyarrow as pa

print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

Pandas: 3.0.3
PyArrow: 25.0.1


In [3]:
# Define the raw and processed data directories

RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
# Load the raw sales, price, and calendar datasets

calendar = pd.read_csv(RAW_DATA_DIR / "calendar.csv")
prices = pd.read_csv(RAW_DATA_DIR / "sell_prices.csv")
sales = pd.read_csv(RAW_DATA_DIR / "sales_train_evaluation.csv")

print("Calendar:", calendar.shape)
print("Prices:", prices.shape)
print("Sales:", sales.shape)

Calendar: (1969, 14)
Prices: (6841121, 4)
Sales: (30490, 1947)


In [5]:
# Calculate demand characteristics for every product-store sales series

id_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

demand_columns = sales.columns[6:]

series_stats = sales[id_columns].copy()

series_stats["total_demand"] = sales[demand_columns].sum(axis=1)
series_stats["avg_daily_demand"] = sales[demand_columns].mean(axis=1)
series_stats["nonzero_days"] = (sales[demand_columns] > 0).sum(axis=1)

series_stats["nonzero_percentage"] = (
    series_stats["nonzero_days"] / len(demand_columns) * 100
)

display(series_stats.head())

,id,item_id,dept_id,cat_id,store_id,state_id,total_demand,avg_daily_demand,nonzero_days,nonzero_percentage
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,633,0.326121,436,22.462648
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,500,0.257599,408,21.020093
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,309,0.159196,229,11.798042
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,3337,1.719217,1316,67.800103
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,1888,0.972694,983,50.643998


In [6]:
# Combine demand activity with historical price variation for each product-store series

price_stats = (
    prices.groupby(["store_id", "item_id"])["sell_price"]
    .agg(
        price_records="count",
        unique_prices="nunique",
        min_price="min",
        max_price="max"
    )
    .reset_index()
)

price_stats["price_range"] = (
    price_stats["max_price"] - price_stats["min_price"]
)

series_stats = series_stats.merge(
    price_stats,
    on=["store_id", "item_id"],
    how="left"
)

display(series_stats.head())

,id,item_id,dept_id,cat_id,store_id,state_id,total_demand,avg_daily_demand,nonzero_days,nonzero_percentage,price_records,unique_prices,min_price,max_price,price_range
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,633,0.326121,436,22.462648,154,3,8.26,9.58,1.32
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,500,0.257599,408,21.020093,262,1,3.97,3.97,0.00
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,309,0.159196,229,11.798042,125,1,2.97,2.97,0.00
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,3337,1.719217,1316,67.800103,277,2,4.34,4.64,0.30
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,1888,0.972694,983,50.643998,266,4,2.48,3.08,0.60


In [7]:
# Summarize demand activity and price variation before selecting modeling series

display(
    series_stats[
        [
            "total_demand",
            "avg_daily_demand",
            "nonzero_days",
            "nonzero_percentage",
            "price_records",
            "unique_prices",
            "price_range"
        ]
    ].describe()
)

print("\nSeries with at least 3 unique prices:",
      (series_stats["unique_prices"] >= 3).sum())

print("Series with at least 5 unique prices:",
      (series_stats["unique_prices"] >= 5).sum())

print("Series with sales on at least 25% of days:",
      (series_stats["nonzero_percentage"] >= 25).sum())

print("Series with sales on at least 50% of days:",
      (series_stats["nonzero_percentage"] >= 50).sum())

,total_demand,avg_daily_demand,nonzero_days,nonzero_percentage,price_records,unique_prices,price_range
count,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000
mean,2195.053231,1.130888,621.163365,32.002234,224.372614,2.769367,0.612636
std,5290.421671,2.725617,433.631478,22.340622,68.000808,1.893627,1.303884
min,15.000000,0.007728,12.000000,0.618238,19.000000,1.000000,0.000000
25%,366.000000,0.188563,262.000000,13.498197,173.000000,1.000000,0.000000
50%,868.000000,0.447192,518.000000,26.687275,260.000000,2.000000,0.280000
75%,2072.000000,1.067491,903.000000,46.522411,282.000000,4.000000,0.720000
max,253859.000000,130.787738,1938.000000,99.845440,282.000000,21.000000,104.060000



Series with at least 3 unique prices: 13847
Series with at least 5 unique prices: 4168
Series with sales on at least 25% of days: 16088
Series with sales on at least 50% of days: 6638


In [8]:
# Select product-store series with sufficient demand activity and price variation

candidate_series = series_stats[
    (series_stats["unique_prices"] >= 3) &
    (series_stats["nonzero_percentage"] >= 25) &
    (series_stats["price_range"] > 0)
].copy()

print("Candidate series:", len(candidate_series))
print(
    "Percentage of all series:",
    round(len(candidate_series) / len(series_stats) * 100, 2),
    "%"
)

print("\nCandidates by category:")
print(candidate_series["cat_id"].value_counts())

print("\nCandidates by store:")
print(candidate_series["store_id"].value_counts())

Candidate series: 9004
Percentage of all series: 29.53 %

Candidates by category:
cat_id
FOODS        5800
HOUSEHOLD    2071
HOBBIES      1133
Name: count, dtype: int64

Candidates by store:
store_id
CA_3    1217
TX_2    1071
CA_1     967
TX_3     928
TX_1     909
WI_3     835
WI_1     819
CA_2     762
CA_4     754
WI_2     742
Name: count, dtype: int64


In [9]:
# Measure historical price variation relative to each product's average price

candidate_series["relative_price_range"] = (
    candidate_series["price_range"] /
    ((candidate_series["min_price"] + candidate_series["max_price"]) / 2)
) * 100

display(
    candidate_series["relative_price_range"]
    .describe()
)

print(
    "\nCandidates with at least 10% relative price variation:",
    (candidate_series["relative_price_range"] >= 10).sum()
)

print(
    "Candidates with at least 20% relative price variation:",
    (candidate_series["relative_price_range"] >= 20).sum()
)

print(
    "Candidates with at least 30% relative price variation:",
    (candidate_series["relative_price_range"] >= 30).sum()
)

count    9004.000000
mean       28.199926
std        26.108445
min         0.503778
25%        11.917404
50%        18.757613
75%        33.613445
max       199.196787
Name: relative_price_range, dtype: float64


Candidates with at least 10% relative price variation: 7527
Candidates with at least 20% relative price variation: 4263
Candidates with at least 30% relative price variation: 2520


In [10]:
# Keep series with sufficient demand activity and meaningful historical price variation

eligible_series = candidate_series[
    candidate_series["relative_price_range"] >= 10
].copy()

print("Eligible series:", len(eligible_series))
print(
    "Percentage of all series:",
    round(len(eligible_series) / len(series_stats) * 100, 2),
    "%"
)

print("\nEligible series by category:")
print(eligible_series["cat_id"].value_counts())

print("\nEligible series by state:")
print(eligible_series["state_id"].value_counts())

Eligible series: 7527
Percentage of all series: 24.69 %

Eligible series by category:
cat_id
FOODS        5219
HOUSEHOLD    1513
HOBBIES       795
Name: count, dtype: int64

Eligible series by state:
state_id
CA    3051
TX    2510
WI    1966
Name: count, dtype: int64


In [11]:
# Filter the wide sales table to product-store series selected for modeling

eligible_keys = eligible_series[["item_id", "store_id"]]

sales_filtered = sales.merge(
    eligible_keys,
    on=["item_id", "store_id"],
    how="inner"
)

print("Original sales shape:", sales.shape)
print("Filtered sales shape:", sales_filtered.shape)

Original sales shape: (30490, 1947)
Filtered sales shape: (7527, 1947)


In [12]:
# Convert filtered daily sales columns from wide format to long format

sales_long = sales_filtered.melt(
    id_vars=[
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id"
    ],
    var_name="d",
    value_name="demand"
)

print("Long-format shape:", sales_long.shape)

display(sales_long.head())

Long-format shape: (14609907, 8)


,id,item_id,dept_id,cat_id,store_id,state_id,d,demand
0,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_008_CA_1_evaluation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,d_1,12
2,HOBBIES_1_020_CA_1_evaluation,HOBBIES_1_020,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_028_CA_1_evaluation,HOBBIES_1_028,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_029_CA_1_evaluation,HOBBIES_1_029,HOBBIES_1,HOBBIES,CA_1,CA,d_1,2


In [13]:
# Merge calendar information into the long-format sales data

calendar_columns = [
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "d",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

pricing_data = sales_long.merge(
    calendar[calendar_columns],
    on="d",
    how="left",
    validate="many_to_one"
)

pricing_data["date"] = pd.to_datetime(pricing_data["date"])

print("Shape after calendar merge:", pricing_data.shape)
print("Missing dates:", pricing_data["date"].isna().sum())

display(pricing_data.head())

Shape after calendar merge: (14609907, 21)
Missing dates: 0


,id,item_id,dept_id,cat_id,store_id,state_id,d,demand,date,wm_yr_wk,...,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,HOBBIES_1_008_CA_1_evaluation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,d_1,12,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,HOBBIES_1_020_CA_1_evaluation,HOBBIES_1_020,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,HOBBIES_1_028_CA_1_evaluation,HOBBIES_1_028,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
4,HOBBIES_1_029_CA_1_evaluation,HOBBIES_1_029,HOBBIES_1,HOBBIES,CA_1,CA,d_1,2,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0


In [14]:
# Merge weekly selling prices using item, store, and calendar week

pricing_data = pricing_data.merge(
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

print("Shape after price merge:", pricing_data.shape)

missing_prices = pricing_data["sell_price"].isna().sum()
missing_percentage = missing_prices / len(pricing_data) * 100

print("Missing prices:", missing_prices)
print("Missing price percentage:", round(missing_percentage, 2), "%")

display(
    pricing_data[
        ["date", "item_id", "store_id", "demand", "sell_price"]
    ].head(10)
)

Shape after price merge: (14609907, 22)
Missing prices: 835583
Missing price percentage: 5.72 %


,date,item_id,store_id,demand,sell_price
0,2011-01-29,HOBBIES_1_005,CA_1,0,NaN
1,2011-01-29,HOBBIES_1_008,CA_1,12,0.46
2,2011-01-29,HOBBIES_1_020,CA_1,0,NaN
3,2011-01-29,HOBBIES_1_028,CA_1,0,6.67
4,2011-01-29,HOBBIES_1_029,CA_1,2,7.44
5,2011-01-29,HOBBIES_1_030,CA_1,0,NaN
6,2011-01-29,HOBBIES_1_032,CA_1,9,0.47
7,2011-01-29,HOBBIES_1_036,CA_1,2,0.96
8,2011-01-29,HOBBIES_1_055,CA_1,0,7.44
9,2011-01-29,HOBBIES_1_058,CA_1,1,7.97


In [15]:
# Check whether missing prices mostly occur on days with zero demand

missing_price_rows = pricing_data[pricing_data["sell_price"].isna()]

print("Missing price rows:", len(missing_price_rows))

print(
    "Missing price rows with zero demand:",
    (missing_price_rows["demand"] == 0).sum()
)

print(
    "Missing price rows with positive demand:",
    (missing_price_rows["demand"] > 0).sum()
)

print(
    "Zero-demand percentage among missing prices:",
    round((missing_price_rows["demand"] == 0).mean() * 100, 2),
    "%"
)

Missing price rows: 835583
Missing price rows with zero demand: 835583
Missing price rows with positive demand: 0
Zero-demand percentage among missing prices: 100.0 %


In [16]:
# Check whether missing prices occur before each product-store's first recorded price

first_price_date = (
    pricing_data[pricing_data["sell_price"].notna()]
    .groupby(["item_id", "store_id"])["date"]
    .min()
    .rename("first_price_date")
)

missing_price_check = missing_price_rows.merge(
    first_price_date,
    on=["item_id", "store_id"],
    how="left"
)

before_first_price = (
    missing_price_check["date"] <
    missing_price_check["first_price_date"]
)

print(
    "Missing rows before first recorded price:",
    before_first_price.sum()
)

print(
    "Percentage before first recorded price:",
    round(before_first_price.mean() * 100, 2),
    "%"
)

display(
    missing_price_check[
        [
            "date",
            "item_id",
            "store_id",
            "demand",
            "first_price_date"
        ]
    ].head(10)
)

Missing rows before first recorded price: 835583
Percentage before first recorded price: 100.0 %


,date,item_id,store_id,demand,first_price_date
0,2011-01-29,HOBBIES_1_005,CA_1,0,2011-05-21
1,2011-01-29,HOBBIES_1_020,CA_1,0,2011-02-12
2,2011-01-29,HOBBIES_1_030,CA_1,0,2012-07-21
3,2011-01-29,HOBBIES_1_090,CA_1,0,2013-06-22
4,2011-01-29,HOBBIES_1_123,CA_1,0,2013-05-18
5,2011-01-29,HOBBIES_1_134,CA_1,0,2011-10-08
6,2011-01-29,HOBBIES_1_153,CA_1,0,2011-12-24
7,2011-01-29,HOBBIES_1_166,CA_1,0,2012-08-04
8,2011-01-29,HOBBIES_1_215,CA_1,0,2011-02-05
9,2011-01-29,HOBBIES_1_225,CA_1,0,2012-06-23


In [17]:
# Remove pre-availability observations that have no recorded selling price

rows_before = len(pricing_data)

pricing_data = pricing_data[
    pricing_data["sell_price"].notna()
].copy()

rows_removed = rows_before - len(pricing_data)

print("Rows before:", rows_before)
print("Rows removed:", rows_removed)
print("Rows remaining:", len(pricing_data))
print("Missing prices remaining:", pricing_data["sell_price"].isna().sum())

Rows before: 14609907
Rows removed: 835583
Rows remaining: 13774324
Missing prices remaining: 0


In [18]:
# Remove merge-only identifiers and organize the cleaned dataset columns

pricing_data = pricing_data.drop(
    columns=["id", "d"]
)

column_order = [
    "date",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI",
    "sell_price",
    "demand"
]

pricing_data = pricing_data[column_order]

print("Cleaned shape:", pricing_data.shape)
display(pricing_data.head())

Cleaned shape: (13774324, 20)


,date,item_id,dept_id,cat_id,store_id,state_id,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,demand
1,2011-01-29,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.46,12
3,2011-01-29,HOBBIES_1_028,HOBBIES_1,HOBBIES,CA_1,CA,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,6.67,0
4,2011-01-29,HOBBIES_1_029,HOBBIES_1,HOBBIES,CA_1,CA,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,7.44,2
6,2011-01-29,HOBBIES_1_032,HOBBIES_1,HOBBIES,CA_1,CA,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.47,9
7,2011-01-29,HOBBIES_1_036,HOBBIES_1,HOBBIES,CA_1,CA,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.96,2


In [19]:
# Optimize data types to reduce memory usage before saving the cleaned dataset

category_columns = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "weekday", "event_name_1", "event_type_1",
    "event_name_2", "event_type_2"
]

for col in category_columns:
    pricing_data[col] = pricing_data[col].astype("category")

pricing_data["wm_yr_wk"] = pd.to_numeric(
    pricing_data["wm_yr_wk"], downcast="integer"
)

for col in ["wday", "month", "year", "snap_CA", "snap_TX", "snap_WI", "demand"]:
    pricing_data[col] = pd.to_numeric(
        pricing_data[col], downcast="integer"
    )

pricing_data["sell_price"] = pd.to_numeric(
    pricing_data["sell_price"], downcast="float"
)

memory_mb = pricing_data.memory_usage(deep=True).sum() / 1024**2

print(f"Optimized memory usage: {memory_mb:.2f} MB")
print("\nData types:")
print(pricing_data.dtypes)

Optimized memory usage: 551.75 MB

Data types:
date            datetime64[us]
item_id               category
dept_id               category
cat_id                category
store_id              category
state_id              category
wm_yr_wk                 int16
weekday               category
wday                      int8
month                     int8
year                     int16
event_name_1          category
event_type_1          category
event_name_2          category
event_type_2          category
snap_CA                   int8
snap_TX                   int8
snap_WI                   int8
sell_price             float32
demand                   int16
dtype: object


In [20]:
# Validate the cleaned dataset before saving it for downstream analysis

print("Dataset shape:", pricing_data.shape)
print("Duplicate rows:", pricing_data.duplicated().sum())

print("\nCritical missing values:")
print(
    pricing_data[
        ["date", "item_id", "store_id", "sell_price", "demand"]
    ].isnull().sum()
)

print("\nInvalid values:")
print("Negative prices:", (pricing_data["sell_price"] < 0).sum())
print("Negative demand:", (pricing_data["demand"] < 0).sum())

print("\nDate range:")
print(pricing_data["date"].min(), "to", pricing_data["date"].max())

print("\nPrice range:")
print(pricing_data["sell_price"].min(), "to", pricing_data["sell_price"].max())

print("\nDemand range:")
print(pricing_data["demand"].min(), "to", pricing_data["demand"].max())

Dataset shape: (13774324, 20)
Duplicate rows: 0

Critical missing values:
date          0
item_id       0
store_id      0
sell_price    0
demand        0
dtype: int64

Invalid values:
Negative prices: 0
Negative demand: 0

Date range:
2011-01-29 00:00:00 to 2016-05-22 00:00:00

Price range:
0.01 to 44.36

Demand range:
0 to 763


In [21]:
# Save the cleaned pricing dataset in an efficient Parquet format

output_path = PROCESSED_DATA_DIR / "pricing_data.parquet"

pricing_data.to_parquet(
    output_path,
    index=False,
    engine="pyarrow"
)

file_size_mb = output_path.stat().st_size / 1024**2

print("Saved to:", output_path)
print(f"File size: {file_size_mb:.2f} MB")

Saved to: ..\data\processed\pricing_data.parquet
File size: 42.10 MB


## Data Cleaning Summary

- Selected product-store series with sufficient demand activity and historical price variation for downstream pricing analysis.
- Converted the selected daily sales data from wide to long format.
- Merged calendar information using the daily M5 identifier.
- Merged historical selling prices using item, store, and week identifiers.
- Missing selling prices were found only before each product-store series' first recorded price and always had zero demand, so these pre-availability observations were removed.
- Removed merge-only identifiers after the datasets were successfully connected.
- Optimized data types to reduce memory usage.
- Performed final integrity checks on critical modeling variables.
- Saved the cleaned dataset in Parquet format for efficient downstream analysis.